In [1]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
%config InlineBackend.figure_format = 'retina'

In [8]:
from pathlib import Path
import numpy as np
from astropy.io import fits
from astropy.table import Table
from matplotlib import pyplot as plt
from kspecdr.inst.isoplane import convert_isoplane_header, add_fiber_table, write_isoplane_converted_image
from kspecdr.io.image import ImageFile
from kspecdr.preproc.make_im import make_im
from kspecdr.preproc.preproc import reduce_bias, reduce_dark, combine_image
from kspecdr.tlm.make_tlm import read_instrument_data, make_tlm
from scipy.signal import find_peaks_cwt, find_peaks
from rascal.util import refine_peaks
from matplotlib.lines import Line2D
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(name)s | %(levelname)s | %(message)s"
    )

WD = Path("/data1/hbahk/kspec/kspecdr")
RESOURCES = WD / "resources"
TESTDIR = RESOURCES / "isoplane_commissioning" / "260128"
CALDIR = TESTDIR / "calib"

In [12]:
logging.getLogger("kspecdr.io.image").setLevel(logging.WARNING)
bias_files = list(CALDIR.glob("Bias*.fits"))
bias_files_converted = [CALDIR / f.name.replace("Bias", "cBias") for f in bias_files]

for bfpath, bfpath_converted in zip(bias_files, bias_files_converted):
    write_isoplane_converted_image(bfpath, bfpath_converted, "BIAS", n_fibers=14)

mbias_path = CALDIR / "mbias.fits"
mbias_file = reduce_bias(bias_files_converted, output_file=mbias_path.as_posix())

2026-01-29 21:23:41,326 | kspecdr.inst.isoplane | INFO | Readout settings: speed=0 MHz, gain=HIGH, noise=LOW
2026-01-29 21:23:41,338 | astropy | WARNING | TimeDeltaMissingUnitWarning: Numerical value without unit or explicit format passed to TimeDelta, assuming days
2026-01-29 21:23:41,339 | kspecdr.inst.isoplane | INFO | Adding fiber table with 14 fibers


2026-01-29 21:23:41,345 | kspecdr.inst.isoplane | INFO | Flipped vertical orientation
2026-01-29 21:23:41,370 | kspecdr.inst.isoplane | INFO | Readout settings: speed=0 MHz, gain=HIGH, noise=LOW
2026-01-29 21:23:41,372 | kspecdr.inst.isoplane | INFO | Adding fiber table with 14 fibers
2026-01-29 21:23:41,377 | kspecdr.inst.isoplane | INFO | Flipped vertical orientation
2026-01-29 21:23:41,399 | kspecdr.inst.isoplane | INFO | Readout settings: speed=0 MHz, gain=HIGH, noise=LOW
2026-01-29 21:23:41,401 | kspecdr.inst.isoplane | INFO | Adding fiber table with 14 fibers
2026-01-29 21:23:41,406 | kspecdr.inst.isoplane | INFO | Flipped vertical orientation
2026-01-29 21:23:41,428 | kspecdr.inst.isoplane | INFO | Readout settings: speed=0 MHz, gain=HIGH, noise=LOW
2026-01-29 21:23:41,430 | kspecdr.inst.isoplane | INFO | Adding fiber table with 14 fibers
2026-01-29 21:23:41,435 | kspecdr.inst.isoplane | INFO | Flipped vertical orientation
2026-01-29 21:23:41,457 | kspecdr.inst.isoplane | INFO |

In [ ]:
flat_files = list(CALDIR.glob("Flat*.fits"))
flat_files_converted = [CALDIR / f.name.replace("Flat", "cFlat") for f in flat_files]

for fpath, fpath_converted in zip(flat_files, flat_files_converted):
    write_isoplane_converted_image(fpath, fpath_converted, "FLAT", n_fibers=14)

for fpath in flat_files_converted:
    make_im(fpath.as_posix(),
        cosmic_ray_method='NONE',
        bias_filename=mbias_path.as_posix(),
        use_bias=True,
        verbose=False)

2026-01-29 21:25:57,074 | kspecdr.inst.isoplane | INFO | Readout settings: speed=0 MHz, gain=HIGH, noise=LOW
2026-01-29 21:25:57,086 | astropy | WARNING | TimeDeltaMissingUnitWarning: Numerical value without unit or explicit format passed to TimeDelta, assuming days
2026-01-29 21:25:57,088 | kspecdr.inst.isoplane | INFO | Adding fiber table with 14 fibers
2026-01-29 21:25:57,095 | kspecdr.inst.isoplane | INFO | Flipped vertical orientation
2026-01-29 21:25:57,140 | kspecdr.inst.isoplane | INFO | Readout settings: speed=0 MHz, gain=HIGH, noise=LOW
2026-01-29 21:25:57,143 | kspecdr.inst.isoplane | INFO | Adding fiber table with 14 fibers
2026-01-29 21:25:57,149 | kspecdr.inst.isoplane | INFO | Flipped vertical orientation
2026-01-29 21:25:57,190 | kspecdr.inst.isoplane | INFO | Readout settings: speed=0 MHz, gain=HIGH, noise=LOW
2026-01-29 21:25:57,193 | kspecdr.inst.isoplane | INFO | Adding fiber table with 14 fibers
2026-01-29 21:25:57,199 | kspecdr.inst.isoplane | INFO | Flipped verti

In [15]:
spec_sets = []
for fpath in flat_files_converted:
    im_path = CALDIR / (fpath.stem + "_im.fits")
    spec_set = fpath.stem[6:13]
    spec_sets.append(spec_set)
    
    fpath_tlm = CALDIR / ("tlm_" + spec_set + ".fits")
    args = {"IMAGE_FILENAME": im_path.as_posix(),
            "TLMAP_FILENAME": fpath_tlm.as_posix()}

    make_tlm(args)

2026-01-29 21:48:09,289 | kspecdr.tlm.make_tlm | INFO | Generating tramline map from /data1/hbahk/kspec/kspecdr/resources/isoplane_commissioning/260128/calib/cFlat_150_620 2026 January 28_im.fits
2026-01-29 21:48:09,367 | kspecdr.tlm.make_tlm | INFO | Instrument code: 99
2026-01-29 21:48:09,368 | kspecdr.tlm.make_tlm | INFO | Starting tramline map generation for non-2DF instrument
2026-01-29 21:48:09,464 | kspecdr.tlm.make_tlm | INFO | Fibres officially in use: 14
2026-01-29 21:48:09,464 | kspecdr.tlm.make_tlm | INFO | Fibres potentially able: 0
2026-01-29 21:48:09,464 | kspecdr.tlm.make_tlm | INFO | Fibres officially dead: 0
2026-01-29 21:48:09,465 | kspecdr.tlm.make_tlm | INFO | Max number of traces: 14
2026-01-29 21:48:09,465 | kspecdr.tlm.make_tlm | INFO | Image dimensions: nspec=1340, nspat=1300
2026-01-29 21:48:09,465 | kspecdr.tlm.make_tlm | INFO | Sweeping image for signs of fibre traces...
2026-01-29 21:48:09,466 | kspecdr.tlm.make_tlm | INFO | Processing column 0/1340 (0.0%)
